# Checking daily case reports for unusual changes

This notebook covers the time-series part of the project. It looks at daily COVID-19 case reports for Uganda and neighboring countries, then flags days that differ sharply from that country's recent reporting pattern. A flag is a prompt to review the data; it does not prove that a value is wrong or predict an outbreak.

**Question:** Which daily case reports stand out from recent patterns, and what might explain them?

## Load and inspect the data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("data/owid/cases_deaths.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Download the OWID cases and deaths CSV and save it to data/owid/cases_deaths.csv; see the README.")
raw = pd.read_csv(DATA_PATH, low_memory=False)
print("Rows and columns:", raw.shape)
print("Available columns:", raw.columns.tolist())
display(raw.head())

## Choose the measure and countries

Check the dataset's metadata for the indicator definition, units, and original data source before interpreting the results. The code uses new cases per million where available and records which field it selected.

In [ ]:
date_col = "date"
place_col = "country" if "country" in raw.columns else ("location" if "location" in raw.columns else None)
if place_col is None or date_col not in raw.columns:
    raise ValueError("Could not find a country/location and date column. Review the OWID file schema.")
candidate_cols = ["cases_new_per_million", "new_cases_per_million", "cases_new", "new_cases"]
value_col = next((c for c in candidate_cols if c in raw.columns), None)
if value_col is None:
    raise ValueError(f"No recognized new-cases column. Available: {raw.columns.tolist()}")

countries = ["Uganda", "Kenya", "Tanzania", "Rwanda"]
df = raw.loc[raw[place_col].isin(countries), [place_col, date_col, value_col]].copy()
df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
df = df.dropna(subset=[date_col]).sort_values([place_col, date_col])
df = df.rename(columns={place_col: "country", date_col: "date", value_col: "reported_cases"})
print("Selected measure:", value_col)
display(df.groupby("country")["reported_cases"].agg(["count", "min", "median", "max"]))
display(df.isna().mean().rename("missing_fraction").to_frame())

## Calculate a rolling anomaly score

Daily reports often have weekday patterns, so first smooth them with a 7-day mean. Each day is then compared with the previous 28 days using the median and median absolute deviation (MAD). The current day is excluded from its own baseline.

In [ ]:
window = 28
threshold = 3.5
scored_parts = []
for country, group in df.groupby("country", sort=False):
    group = group.sort_values("date").copy()
    group["cases_7day"] = group["reported_cases"].rolling(7, min_periods=4).mean()
    previous = group["cases_7day"].shift(1)
    baseline = previous.rolling(window, min_periods=14).median()
    deviation = (previous - baseline).abs().rolling(window, min_periods=14).median()
    scale = (1.4826 * deviation).replace(0, np.nan)
    group["robust_z"] = (group["cases_7day"] - baseline) / scale
    group["flag_for_review"] = group["robust_z"].abs() >= threshold
    scored_parts.append(group)
scored = pd.concat(scored_parts, ignore_index=True)
display(scored.groupby("country")["flag_for_review"].agg(["sum", "count"]))
display(scored.loc[scored.flag_for_review, ["country", "date", "reported_cases", "cases_7day", "robust_z"]].head(20))

## Plot and interpret the flags

In [ ]:
countries_in_data = scored["country"].unique()
fig, axes = plt.subplots(len(countries_in_data), 1, figsize=(12, 3 * len(countries_in_data)), sharex=True)
for ax, country in zip(np.atleast_1d(axes), countries_in_data):
    group = scored[scored.country == country]
    ax.plot(group.date, group.cases_7day, linewidth=1.2, label="7-day mean")
    flags = group[group.flag_for_review]
    ax.scatter(flags.date, flags.cases_7day, color="crimson", s=20, label="Flag for review")
    ax.set_title(country)
    ax.set_ylabel("Reported cases")
    ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

When you see a flagged date, compare it with surrounding days and check the source notes. A spike may reflect a backlog submitted at once; a drop may reflect a delayed or incomplete report. The method also needs a warm-up period before it can score records.

In the report, state the selected indicator, units, date range, threshold, and number of flags. Discuss a few examples and the information you would check before treating them as data-quality problems. These historical reports cannot support a claim that the method predicts outbreaks.

**Source:** Our World in Data COVID-19 documentation: https://docs.owid.io/projects/etl/api/covid/ . Review the current metadata and indicator licenses before use.